In [28]:
from datasets import load_from_disk

In [22]:
import re

def collapse_repeated_tokens_vietnamese(
    text, 
    fillers=None, 
    filler_threshold=2, 
    other_threshold=3
):
    """
    Collapse repeated consecutive tokens in Vietnamese text.
    
    Args:
        text (str): ASR output text.
        fillers (list): List of filler words. Default common Vietnamese particles.
        filler_threshold (int): Min repeat for filler words to collapse.
        other_threshold (int): Min repeat for other words to collapse.
        
    Returns:
        str: Text with repeated tokens collapsed.
    """
    if fillers is None:
        fillers = ["ạ", "ừ", "ờ", "à", "hả", "ơ"]  # danh sách filler words

    collapsed_text = text

    # 1. Collapse filler words
    for filler in fillers:
        pattern = r'\b({f})( \1){{{min_repeat},}}'.format(f=re.escape(filler), min_repeat=filler_threshold-1)
        collapsed_text = re.sub(pattern, r'\1', collapsed_text)
    
    # 2. Collapse other words
    # Regex: (\b\w+\b)( \1){other_threshold-1,} → collapse bất kỳ từ nào lặp liên tiếp >= threshold
    # Loại trừ các filler words đã xử lý
    filler_pattern = '|'.join(re.escape(f) for f in fillers)
    pattern_other = r'\b(?!' + filler_pattern + r'\b)(\S+)\b(?: \1\b){' + str(other_threshold-1) + r',}'
    collapsed_text = re.sub(pattern_other, r'\1', collapsed_text)
    
    return collapsed_text

# -------------------------
# Ví dụ sử dụng
asr_output = ("khi mà cái câu thoải cuối cùng trong trơ lờ này được đưa ra "
              "có phải đấy là thông điệp của bộ phim không ạ ạ ạ ạ ạ ạ "
              "trơ trơ trơ")
collapsed_output = collapse_repeated_tokens_vietnamese(
    asr_output, filler_threshold=3, other_threshold=3
)
print(collapsed_output)


khi mà cái câu thoải cuối cùng trong trơ lờ này được đưa ra có phải đấy là thông điệp của bộ phim không ạ trơ


In [26]:
import re

NUM_DICT = {
    0: "không", 1: "một", 2: "hai", 3: "ba", 4: "bốn", 5: "năm",
    6: "sáu", 7: "bảy", 8: "tám", 9: "chín"
}

UNIT = ["", "nghìn", "triệu", "tỷ"]

# -------------------------
# 1. Number → Vietnamese text
def number_to_vietnamese(n):
    if n == 0:
        return NUM_DICT[0]

    parts = []
    str_n = str(n)
    while len(str_n) % 3 != 0:
        str_n = "0" + str_n

    groups = [str_n[i:i+3] for i in range(0, len(str_n), 3)]
    group_count = len(groups)

    for i, g in enumerate(groups):
        hundreds = int(g[0])
        tens = int(g[1])
        units = int(g[2])
        group_text = []

        # hundreds
        if hundreds > 0:
            group_text.append(NUM_DICT[hundreds] + " trăm")
        elif i != 0:
            group_text.append("không trăm")

        # tens
        if tens > 1:
            group_text.append(NUM_DICT[tens] + " mươi")
            if units == 1:
                group_text.append("mốt")
            elif units == 5:
                group_text.append("lăm")
            elif units != 0:
                group_text.append(NUM_DICT[units])
        elif tens == 1:
            group_text.append("mười")
            if units == 1:
                group_text.append("một")
            elif units == 5:
                group_text.append("lăm")
            elif units != 0:
                group_text.append(NUM_DICT[units])
        elif tens == 0:
            if units != 0:
                group_text.append(NUM_DICT[units])

        # unit
        unit_idx = group_count - i - 1
        if group_text and UNIT[unit_idx]:
            group_text.append(UNIT[unit_idx])

        parts.append(" ".join(group_text))

    return " ".join([p for p in parts if p]).replace("  ", " ").strip()

def convert_numbers_in_text(text):
    """Convert number literals in text → Vietnamese words"""
    def repl(m):
        num = int(m.group())
        return number_to_vietnamese(num)
    return re.sub(r'\b\d+\b', repl, text)


# -------------------------
# 2. Vietnamese text → Number
VI_NUM_MAP = {
    "không":0, "một":1, "mốt":1, "hai":2, "ba":3, "bốn":4, "tư":4, "năm":5, "lăm":5,
    "sáu":6, "bảy":7, "tám":8, "chín":9
}

UNIT_MAP = {
    "trăm":100, "nghìn":1000, "triệu":1000000, "tỷ":1000000000
}

def vietnamese_to_number(text):
    """
    Convert Vietnamese number words → numeric literal
    Supports integers up to billions
    """
    words = text.split()
    total = 0
    group = 0
    current = 0

    for w in words:
        if w in VI_NUM_MAP:
            current += VI_NUM_MAP[w]
        elif w in UNIT_MAP:
            unit_val = UNIT_MAP[w]
            if unit_val >= 1000:
                group = (group + current) * unit_val
                total += group
                group = 0
                current = 0
            else:  # trăm
                current *= unit_val
        else:
            # non-number word, reset group
            group += current
            total += group
            group = 0
            current = 0

    total += group + current
    return str(total)

def convert_vietnamese_numbers_in_text(text):
    """
    Detect sequences of Vietnamese number words and convert → numeric literal
    """
    # Simple regex to detect number word sequences
    num_words = "|".join(VI_NUM_MAP.keys()) + "|" + "|".join(UNIT_MAP.keys())
    pattern = r'\b(?:' + num_words + r')(?:\s+(?:' + num_words + r'))*\b'

    def repl(m):
        return vietnamese_to_number(m.group())

    return re.sub(pattern, repl, text)


# -------------------------
# Ví dụ sử dụng
asr_output = "cách đây mười năm đồng đô la có giá khoảng mười lăm nghìn tám trăm đồng"
print("ASR Output:", asr_output)
print("Text → Number:")
print(convert_vietnamese_numbers_in_text(asr_output))
# Output: "cách đây 10 năm đồng đô la có giá khoảng 15800 đồng"

asr_output2 = "cách đây 10 năm đồng đô la có giá khoảng 15800 đồng"
print("ASR Output:", asr_output2)
print("Number → Text:")
print(convert_numbers_in_text(asr_output2))
# Output: "cách đây mười năm đồng đô la có giá khoảng mười lăm nghìn tám trăm đồng"


ASR Output: cách đây mười năm đồng đô la có giá khoảng mười lăm nghìn tám trăm đồng
Text → Number:
cách đây mười 5 đồng đô la có giá khoảng mười 5800 đồng
ASR Output: cách đây 10 năm đồng đô la có giá khoảng 15800 đồng
Number → Text:
cách đây mười năm đồng đô la có giá khoảng mười lăm nghìn tám trăm đồng


In [27]:
VI_NUM_MAP = {
    "không":0, "một":1, "mốt":1, "hai":2, "ba":3, "bốn":4, "tư":4, "năm":5, "lăm":5,
    "sáu":6, "bảy":7, "tám":8, "chín":9, "mười":10
}

UNIT_MAP = {
    "trăm":100, "nghìn":1000, "triệu":1000000, "tỷ":1000000000
}

def vietnamese_number_to_int(words):
    """
    Convert list of Vietnamese number words to integer.
    Example: ["mười", "lăm", "nghìn", "tám", "trăm"] -> 15800
    """
    total = 0
    current_group = 0
    current_number = 0

    i = 0
    while i < len(words):
        w = words[i]
        if w in VI_NUM_MAP:
            val = VI_NUM_MAP[w]
            if val == 10 and (i+1 < len(words)) and words[i+1] in VI_NUM_MAP:
                # Handle "mười lăm" -> 15
                current_number = 10 + VI_NUM_MAP[words[i+1]]
                i += 1
            else:
                current_number = val
        elif w in UNIT_MAP:
            unit_val = UNIT_MAP[w]
            if unit_val == 100:
                current_number *= 100
            else:  # nghìn, triệu, tỷ
                current_group += current_number
                current_group *= unit_val
                total += current_group
                current_group = 0
                current_number = 0
        else:
            # not a number word
            current_group += current_number
            total += current_group
            current_group = 0
            current_number = 0
        i += 1

    total += current_group + current_number
    return str(total)

def convert_vietnamese_numbers_in_text(text):
    """
    Detect sequences of Vietnamese number words and convert → numeric literal
    """
    num_words = list(VI_NUM_MAP.keys()) + list(UNIT_MAP.keys())
    pattern = r'\b(?:' + '|'.join(num_words) + r')(?:\s+(?:' + '|'.join(num_words) + r'))*\b'

    def repl(m):
        words = m.group().split()
        return vietnamese_number_to_int(words)

    return re.sub(pattern, repl, text)

# -------------------------
# Ví dụ sử dụng
asr_output = "cách đây mười năm đồng đô la có giá khoảng mười lăm nghìn tám trăm đồng"
converted = convert_vietnamese_numbers_in_text(asr_output)
print(converted)


cách đây 15 đồng đô la có giá khoảng 15800 đồng


In [7]:
from underthesea import pos_tag

# Hàm align_sequences đơn giản theo token
def align_sequences(ref_tokens, hyp_tokens):
    alignments = []
    min_len = min(len(ref_tokens), len(hyp_tokens))
    
    # So sánh phần trùng
    for i in range(min_len):
        if ref_tokens[i] == hyp_tokens[i]:
            alignments.append(("C", ref_tokens[i], hyp_tokens[i]))
        else:
            alignments.append(("S", ref_tokens[i], hyp_tokens[i]))
    
    # Xử lý thừa token
    if len(ref_tokens) > len(hyp_tokens):
        for r in ref_tokens[min_len:]:
            alignments.append(("D", r, "---"))
    elif len(hyp_tokens) > len(ref_tokens):
        for h in hyp_tokens[min_len:]:
            alignments.append(("I", "---", h))
    
    return alignments

# Sample data
samples = [
    ("Thủ đô Hà Nội", "thủ đô hà nội"),
    ("Học sinh đi học rất chăm", "Học sinh đi học chăm"),
    ("Tôi yêu lập trình Python", "Tôi yêu lập trình")
]

word_type_errors = {}

for idx, (ref, hyp) in enumerate(samples, 1):
    print(f"\n=== Sample {idx} ===")
    print("Ref :", ref)
    print("Hyp :", hyp)
    
    # POS tagging trên ref
    pos_tags = pos_tag(ref)  # [(token, tag)]
    ref_tokens = [t for t, _ in pos_tags]
    hyp_tokens = hyp.split()  # prediction vẫn split bình thường

    alignments = align_sequences(ref_tokens, hyp_tokens)
    
    for i, (op, r, h) in enumerate(alignments):
        tag = pos_tags[i][1] if i < len(pos_tags) else "UNK"
        status = {"C": "✔", "S": "S", "D": "D", "I": "I"}.get(op, "?")
        print(f"{i+1:2d}: {r:<15} | {h:<15} | {op} | {status} | POS: {tag}")
        
        # Cập nhật thống kê lỗi theo loại từ
        if op in ("S", "D", "I"):
            word_type_errors[tag] = word_type_errors.get(tag, 0) + 1

print("\n=== Tổng hợp lỗi theo loại từ ===")
for tag, count in word_type_errors.items():
    print(f"{tag}: {count}")



=== Sample 1 ===
Ref : Thủ đô Hà Nội
Hyp : thủ đô hà nội
 1: Thủ đô          | thủ             | S | S | POS: N
 2: Hà Nội          | đô              | S | S | POS: Np
 3: ---             | hà              | I | I | POS: UNK
 4: ---             | nội             | I | I | POS: UNK

=== Sample 2 ===
Ref : Học sinh đi học rất chăm
Hyp : Học sinh đi học chăm
 1: Học sinh        | Học             | S | S | POS: N
 2: đi              | sinh            | S | S | POS: V
 3: học             | đi              | S | S | POS: V
 4: rất             | học             | S | S | POS: R
 5: chăm            | chăm            | C | ✔ | POS: A

=== Sample 3 ===
Ref : Tôi yêu lập trình Python
Hyp : Tôi yêu lập trình
 1: Tôi             | Tôi             | C | ✔ | POS: P
 2: yêu             | yêu             | C | ✔ | POS: V
 3: lập trình       | lập             | S | S | POS: N
 4: Python          | trình           | S | S | POS: Np

=== Tổng hợp lỗi theo loại từ ===
N: 3
Np: 2
UNK: 2
V: 2
R: 1


In [5]:
from underthesea import pos_tag

sentence = "Hà Nội là thủ đô"
tokens_pos = pos_tag(sentence)
print(tokens_pos)

[('Hà Nội', 'Np'), ('là', 'V'), ('thủ đô', 'N')]


In [10]:
from underthesea import pos_tag

def align_sequences_pos(ref, hyp):
    """
    Align ref/hyp tokens, ghép hyp tokens theo ref POS token.
    Trả về list: (op, ref_token, hyp_token, pos_tag)
    """
    ref_tokens_pos = pos_tag(ref)  # [(token, POS)]
    hyp_tokens = hyp.split()
    
    hyp_idx = 0
    alignments = []
    
    for ref_tok, tag in ref_tokens_pos:
        n_words = len(ref_tok.split())  # số từ trong token ref
        hyp_tok = " ".join(hyp_tokens[hyp_idx:hyp_idx+n_words]) if hyp_idx < len(hyp_tokens) else "---"
        hyp_idx += n_words
        
        if hyp_tok == ref_tok:
            op = "C"
        elif hyp_tok == "---":
            op = "D"
        else:
            op = "S"
        
        alignments.append((op, ref_tok, hyp_tok, tag))
    
    # Phần còn lại của hyp → Insert
    while hyp_idx < len(hyp_tokens):
        alignments.append(("I", "---", hyp_tokens[hyp_idx], "UNK"))
        hyp_idx += 1
    
    return alignments

# Example
ref = "Thủ đô Hà Nội"
hyp = "thủ đô hà nội"
alignments = align_sequences_pos(ref, hyp)
for i, (op, r, h, tag) in enumerate(alignments, 1):
    print(f"{i:2d}: {r:<10} | {h:<10} | {op} | POS: {tag}")


 1: Thủ đô     | thủ đô     | S | POS: N
 2: Hà Nội     | hà nội     | S | POS: Np


In [15]:
pos_tag("Hà Nội")

[('Hà Nội', 'Np')]

In [9]:
ref = "Thủ đô Hà Nội"
hyp = "thủ đô hà nội"

alignments = align_tokens_pos(ref, hyp)
for idx, (op, r, h, tag) in enumerate(alignments, 1):
    print(f"{idx:2d}: {r:<10} | {h:<10} | {op} | POS: {tag}")


 1: Thủ đô     | thủ đô hà  | S | POS: N
 2: Hà Nội     | nội        | S | POS: Np
